# 本次实验修改说明
按用户要求用 **large** 替代PDF的2.7B：d_model=1280、d_ff=5120、36层、20头。
本次(a)使用context=512、batch=1、FP32，分别测inference/train；原讲义中2.7B的要求与数值仅供对照，不是本次large结果。
(d)本次单个残差流激活为[1,512,1280]，FP32为2.5 MiB。
已补齐memory_profile的记录、峰值统计与finally停止逻辑；run_model沿用当前文件。

large/batch4/context512训练仍OOM，按用户授权改为batch1。原题参考batch=4；本次是替代实验。


# 作业四：CUDA 显存分析——手动修改练习

基于作业1的 `run_model()` 和 `memory_profile()` 改造，不修改作业1。这里的作业四是PDF §1.1.6 memory profiling，包含(a)–(e)。
提供可运行的模型基础部分及待完成的分析函数；请逐步插入讲解中的代码，不要直接“运行全部”期待得到完整报告。

正式目标：2.7B，batch=4、词表10000、上下文128/256/512，FP32及BF16 autocast；比较仅推理前向和完整训练（前向+loss+反向+AdamW）。
**你的RTX3090 24GB此前在2.7B反向已经OOM，加入AdamW状态会更吃显存。** small只用于学习方法，不能冒充2.7B结果。无法完成的正式配置需记录限制；完成全部2.7B训练实验需要更大显存设备，不能保证本机能跑通。

| 小问 | 需要获得什么 |
|---|---|
| (a) | context=512的推理/完整训练两张Active Memory Timeline截图，2–3句解释 |
| (b) | context=128/256/512，推理及完整训练的峰值显存表 |
| (c) | BF16混合精度的相同峰值，对比FP32，2–3句解释 |
| (d) | 一个残差流激活张量的FP32大小及推导，1–2句 |
| (e) | 最大单次分配大小、调用栈和来源，1–2句 |

PDF正文包含optimizer step；不要采用某些notebook转述中“不含optimizer”的误写。

## 1. 原代码先改什么

1. 你粘贴的两段代码缩进混在一起：`memory_profile`应是独立函数，不应嵌在`run_model`里；`run`是传入的回调，不能在`run_model`里无定义调用。
2. 原 `run_model()` 每次被调用都创建模型和优化器。改为在外面创建一次，`run_model()`只执行一步。否则预热和正式分析不是同一个模型，图中会混入加载和优化器初始化。
3. `del X, attention_fn, out, loss` 中前三个变量不存在；删除语句在第一个未知变量处失败，根本没有可靠清理。改为finally清理实际对象。
4. 去掉学习率调度、梯度裁剪及测量区间打印：本题不要求收敛，额外操作会改变显存峰值。使用随机token即可。
5. `torch.profiler`耗时表和`export_stacks`不是显存时间线；本题主要使用CUDA memory history、snapshot和峰值计数器。`svg_path`变量本身不会生成SVG。
6. 原来的额外10次warmup不必保留。统一用一个明确的预热次数，并在预热后重置峰值。
7. 正式参数与原来的6层、batch32、词表50000不同，请按PDF配置。

In [1]:
import gc
import inspect
from contextlib import nullcontext
from pathlib import Path
import torch
import torch.nn.functional as F
from cs336_basics.model import BasicsTransformerLM
from cs336_basics.optimizer import AdamW

assert torch.cuda.is_available(), "请选择项目GPU内核"
print(torch.__version__, torch.cuda.get_device_name(0))
print("工作目录:", Path.cwd())
print("memory history接口:", inspect.signature(torch.cuda.memory._record_memory_history))

2.9.1+cu128 NVIDIA GeForce RTX 3090
工作目录: /home/bygpu/projects/diy-llm/coursework/assignment2-systems/cs336_systems
memory history接口: (enabled: Optional[str] = 'all', context: Optional[str] = 'all', stacks: str = 'all', max_entries: int = 9223372036854775807, device: 'Device' = None, clear_history: bool = False, compile_context: bool = False, global_record_annotations: bool = False)


## 2. 参数区：先small调试，再切换正式2.7B

每次只运行一种(size, context, mode, precision)配置，保存输出后重启内核再换下一组，避免异常traceback或其他对象影响显存。
不要同时运行之前的性能测试；用终端 `nvidia-smi` 查看其他进程。

BF16只通过autocast开启，模型参数保持FP32，不使用`model.bfloat16()`。FP32和BF16比较时使用完全相同的模型/输入尺寸。

In [ ]:
SIZE = "large"  # 按用户要求替代2.7B；非原题规模
CONTEXT_LENGTH = 512  # (a)要求的上下文长度
MODE = "inference"  # 第二次实验改为train并重建模型
USE_BF16 = False  # (a) FP32基线
NUM_WARMUPS = 1            # 保持各组一致，训练预热会初始化AdamW状态
BATCH_SIZE = 1  # 用户授权：large/context512训练需减小batch
VOCAB_SIZE = 10000

CONFIGS = {
    "large": dict(d_model=1280, d_ff=5120, num_layers=36, num_heads=20),
    "small": dict(d_model=768, d_ff=3072, num_layers=12, num_heads=12),
    "2.7B": dict(d_model=2560, d_ff=10240, num_layers=32, num_heads=32),
}
assert SIZE in CONFIGS and MODE in ("inference", "train")
assert NUM_WARMUPS >= 1
precision = "bf16" if USE_BF16 else "fp32"
description = f"{SIZE}_ctx{CONTEXT_LENGTH}_{MODE}_{precision}_batch{BATCH_SIZE}"
snapshot_dir = Path("memory_snapshots_assignment4")
snapshot_dir.mkdir(exist_ok=True)
snapshot_path = snapshot_dir / f"{description}.pickle"
print("配置:", description)
print("快照路径:", snapshot_path.resolve())

## 3. 把模型初始化移到外面（提供可运行基础）

这格只创建模型、输入和优化器，不开始正式分析。若此处OOM，记录失败阶段为“模型加载”。重启内核恢复。
首次运行没有残留时可直接执行。不要重复运行这格来更换模型；请重启内核后从参数区重新开始。

In [18]:
torch.manual_seed(42)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

model = BasicsTransformerLM(
    vocab_size=VOCAB_SIZE,
    context_length=CONTEXT_LENGTH,
    rope_theta=10000.0,
    **CONFIGS[SIZE],
)
model = model.to(device="cuda", dtype=torch.float32)
model.train(MODE == "train")
x = torch.randint(VOCAB_SIZE, (BATCH_SIZE, CONTEXT_LENGTH), device="cuda")
y = torch.randint(VOCAB_SIZE, x.shape, device="cuda")
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=0.1) if MODE == "train" else None
print("加载完成，已分配MiB:", torch.cuda.memory_allocated() / 1024**2)

加载完成，已分配MiB: 13130.947265625


## 4. 改造run_model：一次调用只执行一步

下面保留你原来的前向、loss、反向和AdamW，去掉模型创建、学习率调度和错误清理。
推理必须no_grad；否则会保存反向激活，不能当作“仅推理”显存。
`stage_memory`只是阶段结束时显存，不能代替全程峰值；它用于解释时间线。

In [19]:
def run_model():
    logits = loss = None
    stage_memory = {}
    model.zero_grad(set_to_none=True)
    context = (
        torch.autocast(device_type="cuda", dtype=torch.bfloat16)
        if USE_BF16 else nullcontext()
    )
    try:
        with torch.set_grad_enabled(MODE == "train"): # true时建立计算图，否则推理模式不建立计算图， set_grad_enabled功能
            with context:
                logits = model(x)
                torch.cuda.synchronize()
                stage_memory["after_forward_MiB"] = torch.cuda.memory_allocated() / 1024**2
                if MODE == "train": # 控制要不要算loss
                    loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))

        if MODE == "train":
            loss.backward()  # autocast外反向
            torch.cuda.synchronize()
            stage_memory["after_backward_MiB"] = torch.cuda.memory_allocated() / 1024**2
            optimizer.step()
            torch.cuda.synchronize()
            stage_memory["after_optimizer_MiB"] = torch.cuda.memory_allocated() / 1024**2
        return stage_memory
    finally:
        # 不返回GPU张量，不保留本轮计算图。
        logits = None
        loss = None

## 5. 手动完成memory_profile的三个TODO

下面函数已保留预热和安全退出结构，但尚未生成快照。请先阅读，再把下方三个片段放到对应位置。
预热后清空梯度，但保留AdamW状态，这是**稳态完整训练**的基线；FP32与BF16必须采用同样策略。
不要把`empty_cache()`放进被测步骤，它不能释放仍被引用的张量，也会改变缓存行为。

In [ ]:
def memory_profile(description, run, num_warmups=1):
    # Warmup initializes the existing model and AdamW state, not a new model.
    for _ in range(num_warmups):
        run()
    torch.cuda.synchronize()
    model.zero_grad(set_to_none=True)
    gc.collect()
    torch.cuda.synchronize()
    path = snapshot_dir / f"{description}.pickle"
    recording = False
    try:
        torch.cuda.memory._record_memory_history(max_entries=1000000)
        recording = True
        torch.cuda.reset_peak_memory_stats()
        baseline_bytes = torch.cuda.memory_allocated()
        stage_memory = run()
        torch.cuda.synchronize()
        peak_bytes = torch.cuda.max_memory_allocated()
        peak_reserved_bytes = torch.cuda.max_memory_reserved()
        torch.cuda.memory._dump_snapshot(str(path))
        result = {
            "configuration": description,
            "baseline_MiB": baseline_bytes / 1024**2,
            "peak_allocated_MiB": peak_bytes / 1024**2,
            "peak_reserved_MiB": peak_reserved_bytes / 1024**2,
            "extra_above_baseline_MiB": (peak_bytes-baseline_bytes) / 1024**2,
            "snapshot": str(path.resolve()),
            "stage_end_memory": stage_memory,
        }
        print(result, flush=True)
        return result
    except torch.cuda.OutOfMemoryError:
        if recording:
            torch.cuda.memory._dump_snapshot(str(path.with_suffix(".oom.pickle")))
        raise
    finally:
        if recording:
            torch.cuda.memory._record_memory_history(enabled=None)


**TODO 1** 放在`try`之前，保持函数内缩进：

```python
torch.cuda.memory._record_memory_history(max_entries=1000000)
recording = True
torch.cuda.reset_peak_memory_stats()
baseline_bytes = torch.cuda.memory_allocated()
```

**TODO 2** 放在run和同步之后，保持try内缩进：

```python
peak_bytes = torch.cuda.max_memory_allocated()
peak_reserved_bytes = torch.cuda.max_memory_reserved()
torch.cuda.memory._dump_snapshot(str(path))
result = {
    "configuration": description,
    "baseline_MiB": baseline_bytes / 1024**2,
    "peak_allocated_MiB": peak_bytes / 1024**2,
    "peak_reserved_MiB": peak_reserved_bytes / 1024**2,
    "extra_above_baseline_MiB": (peak_bytes - baseline_bytes) / 1024**2,
    "snapshot": str(path.resolve()),
    "stage_end_memory": stage_memory,
}
print(result)
```

**TODO 3** 替换finally中的pass：

```python
if recording:
    torch.cuda.memory._record_memory_history(enabled=None)
```

区别：`peak_allocated`是PyTorch张量分配峰值，真实的模型显存占用，Tensor（比如权重、激活值、梯度）占用的字节数。`peak_reserved`包含闲置缓存；`nvidia-smi`还包含其他进程和非PyTorch显存。PyTorch 的显存分配器（Caching Allocator）很聪明，它释放 Tensor 后，不会立刻把显存还给显卡，而是“扣留”下来作为缓存（缓存池），以备下次直接用。所以 reserved 永远大于等于 allocated。正式表格主要报告`peak_allocated`，不要用“峰值减基线”的额外显存冒充模型总峰值。
这些带下划线的memory API属于内部接口，版本之间可能变化；上方环境格会显示本机签名。

先在small上补完并运行；快照确认存在后再做正式配置。

In [21]:
# 完成三个TODO后才执行；未完成时返回None且不会生成快照。
result = memory_profile(description, run_model, num_warmups=NUM_WARMUPS)
if result is None:
    print("请先完成memory_profile中的TODO 1、2、3。")

OutOfMemoryError: CUDA out of memory. Tried to allocate 128.00 MiB. GPU 0 has a total capacity of 23.56 GiB of which 69.88 MiB is free. Process 1903 has 260.00 MiB memory in use. Including non-PyTorch memory, this process has 22.79 GiB memory in use. Of the allocated memory 22.28 GiB is allocated by PyTorch, and 219.87 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 6. (a) 打开Active Memory Timeline并截图

1. FP32、2.7B、context=512、MODE=inference生成一份快照。
2. 关闭/重启内核，改MODE=train生成另一份；train必须包含AdamW。
3. 浏览器打开 https://pytorch.org/memory_viz ，把pickle拖入页面。
4. 若浏览器在本机Windows，先在VS Code远程文件树下载pickle；若在向日葵Ubuntu浏览器，可直接从远程目录拖入。
5. 选择 **Active Memory Timeline**，放大正式单步，截图并在图注标清模型、context、模式、精度和单位。

Linux查看快照：

```bash
cd ~/projects/diy-llm/coursework/assignment2-systems/cs336_systems
ls -lh memory_snapshots_assignment4
```

根据图中分配/释放、调用栈和`stage_end_memory`解释前向激活增长、反向释放及梯度占用、优化器临时张量。稳态AdamW状态在预热时已建立，不应误认为它会在正式步骤中首次出现。
不要仅根据单个尖峰形状断言阶段；用分配调用栈佐证。`record_function`/NVTX不是此网页的阶段时间轴。

**2–3句模板（填实测）：**推理时间线显示______；完整训练中______，结合______调用栈可辨认______阶段。两种模式峰值分别为______、______。

若2.7B训练OOM：提交真实限制和OOM诊断，不能拿small训练截图当2.7B完整截图。补齐完整交付需要足够显存的设备。

## 7. (b) 三种上下文长度的峰值显存

固定2.7B/FP32，依次运行上下文128、256、512的inference和train，共6组；每组重启内核，确保没有别的模型驻留。
每组记录`peak_allocated_MiB`。截图任务只要求(a)的512，但表格三种长度都要测。

| context | FP32推理峰值/MiB | FP32完整训练峰值/MiB | 备注 |
|---:|---:|---:|---|
| 128 | 待填 | 待填 | |
| 256 | 待填 | 待填 | |
| 512 | 待填 | 待填 | |

峰值应包含常驻模型参数、测量时已有优化器状态等，不只统计激活。内存单位统一按字节/1024²，PDF称MB，实际是MiB。
OOM填OOM并说明发生于加载/预热/正式步骤，不能把OOM前的局部峰值当作成功结果。

## 8. (c) BF16混合精度显存对比

把`USE_BF16=True`，其余设置不变，重复推理和完整训练。建议同样测三种长度以便与(b)逐行比较。
模型参数保持FP32；autocast主要改变部分算子及激活精度，**显存不保证减半**，权重转换缓存还可能增加某些阶段开销。

| context | FP32推理/MiB | BF16推理/MiB | 推理节省/% | FP32训练/MiB | BF16训练/MiB | 训练节省/% |
|---:|---:|---:|---:|---:|---:|---:|
| 128 | | | | | | |
| 256 | | | | | | |
| 512 | | | | | | |

节省百分比=`(FP32峰值-BF16峰值)/FP32峰值*100%`。负值表示BF16占用反而更多，不修改或删除负值。

**2–3句模板：**在context=______时，混合精度使推理峰值从______变为______，完整训练峰值从______变为______。这种变化表明______；参数、梯度和AdamW状态仍主要采用FP32，因此______。
如训练两组都OOM，不计算节省比例，也不声称混合精度解决了OOM。

## 9. (d) 单个残差流激活张量大小

不是模型总显存，也不是把每层所有激活相加。一个残差流张量形状是`[B,T,d_model]`，FP32每元素4字节：

`大小 = B*T*d_model*4 / 1024**2 MiB`。

参考B=4、2.7B的d_model=2560：T=128/256/512时，分别为5/10/20 MiB。
context=512的提交答案可写：单个激活形状为`[4,512,2560]`，大小为`4*512*2560*4/1024²=20 MiB`。这仅是一个张量，不包含梯度、attention矩阵、参数或其他层的激活。

In [ ]:
for context in (128, 256, 512):
    size_mib = BATCH_SIZE * context * CONFIGS[SIZE]['d_model'] * 4 / 1024**2
    print(f"context={context}: 单个FP32残差激活 {size_mib:.1f} MiB")

## 10. (e) 最大分配的大小和来源

打开2.7B推理的Active Memory Timeline，把Detail调低（例如10%表示保留较大的分配，不是保留最小10%）。
点击最大分配条，记录精确大小及allocation stack；沿调用栈定位embedding、输出投影、attention分数、FFN或其他操作，不能预先认定一定是attention。

| context/precision | 最大单次分配/MiB | 张量形状/候选 | 实际调用栈位置 | 解释 |
|---|---:|---|---|---|
| 512/FP32 | | | | |

用`元素数*每元素字节`交叉验证大小；如果只看到分配器segment，不要把整个segment当成单个张量。

**特别注意历史起点：**主实验在warmup后才开始记录，之前创建的权重/优化器状态仍计入总显存，但可能没有可追溯的分配栈。
如果最大的条属于记录前已存在的分配、调用栈缺失，请另做一次“诊断快照”：重启内核，在第3节创建模型之前开始history，然后正常创建模型和预热；第5节TODO1不要重复开启history，只重置峰值。保存后停止history。
这样能追踪权重来源。诊断文件应改名注明`from_init`，(a)仍截图正式单步范围，(b)(c)仍使用warmup后重置的峰值，不能把初始化峰值混入。

不要盲目照搬以前报告的最大张量结论；(e)必须来自自己的快照。

**1–2句模板：**最大分配约______MiB，调用栈指向______；结合形状______，它对应______。

## 11. 可选：给原PyTorch Profiler表增加显存列

这是辅助学习，不代替memory_viz：

```python
with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU, torch.profiler.ProfilerActivity.CUDA],
    profile_memory=True,
    record_shapes=True,
    with_stack=True,
) as prof:
    run_model()
    torch.cuda.synchronize()
print(prof.key_averages().table(sort_by="self_cuda_memory_usage", row_limit=20))
```

在预热后独立运行，不要把这个profiler与峰值/快照测量混在同一轮，避免记录额外开销。
算子显存列通常是分配/释放的归属统计，不能将各行相加当成峰值；父子事件也可能重复统计。
若字段名在当前版本变更，查看表头使用对应device memory字段。

## 12. 结束后手动清理

先保存快照与结果，再执行下格；模型被清理后不能继续调用run_model，下一组从重启内核开始。
发生OOM后推荐直接重启内核，traceback可能保留其他引用；`empty_cache`不能强制删除活张量。

In [ ]:
torch.cuda.memory._record_memory_history(enabled=None)
if "model" in globals() and model is not None:
    model.zero_grad(set_to_none=True)
model = optimizer = x = y = None
gc.collect()
torch.cuda.empty_cache()
print("已分配MiB:", torch.cuda.memory_allocated() / 1024**2)

## 提交检查

- (a) 两张真实2.7B/context512时间线截图，标明精度/推理或完整训练，2–3句解释。
- (b) 三种长度各两个峰值，单位与口径一致；失败写OOM。
- (c) BF16与FP32的匹配结果、2–3句讨论。
- (d) 单个残差激活的形状、公式与MiB大小。
- (e) 实际最大分配及调用栈证据，1–2句解释。
- 记录GPU、PyTorch版本、预热次数和AdamW稳态/首次更新策略。

参考：[PyTorch memory snapshot](https://docs.pytorch.org/docs/stable/torch_cuda_memory)、[CUDA显存计数](https://docs.pytorch.org/docs/stable/notes/cuda.html#memory-management)、[memory_viz](https://pytorch.org/memory_viz)。

## (a)、(d) 本次large替代实验实测结果

配置：large(d_model=1280,d_ff=5120,36层,20头)，FP32，context=512，batch=1，vocab=10000；预热1步后采集1步。原题2.7B/batch4未满足，此为用户授权的替代实验。

| 模式 | 基线/MiB | 峰值allocated/MiB | 峰值reserved/MiB |
|---|---:|---:|---:|
| inference | 3814.271 | 3907.067 | 3990.000 |
| train | 11426.421 | 15829.898 | 17728.000 |

(a) 推理仅保留常驻权重和短暂前向工作空间，时间线上各层重复出现较小的分配/释放峰值。完整训练在AdamW预热后已有优化器状态，前向保存反向激活使显存明显增长；反向释放激活的同时产生参数梯度，随后AdamW更新产生临时分配。不能仅凭一个尖峰断言阶段，应结合分配栈与下方阶段结束时显存确认。

![large batch1 inference](memory_snapshots_assignment4/large_inference_batch1.png)

![large batch1 train](memory_snapshots_assignment4/large_train_batch1.png)

(d) 单个残差流激活形状[1,512,1280]，FP32每元素4字节：1*512*1280*4/1024²=2.5 MiB；不含其他层、梯度、参数或attention矩阵。

补充：large/batch4/context512的完整训练仍OOM；batch1训练使用PYTORCH_ALLOC_CONF=expandable_segments:True，推理使用默认分配器，因此reserved峰值不宜用作纯模式对比；主表使用allocated。快照在预热后开启，参数与优化器常驻状态可能没有原始分配调用栈。截图来自PyTorch官方Active Memory Timeline渲染器，横向为分配事件序列，不能直接当作毫秒时间轴。

阶段结束时显存（MiB，不是各阶段峰值）：

```json
{
  "inference": {
    "after_forward_MiB": 3834.2705078125
  },
  "train": {
    "after_forward_MiB": 15684.8134765625,
    "after_backward_MiB": 15247.18408203125,
    "after_optimizer_MiB": 15245.18408203125
  }
}
```


# 作业四(b)(c)实测结果

配置：large(d_model=1280,d_ff=5120,36层,20头)，batch=1、vocab=10000、FP32参数；每组独立进程，预热1步后测1步。使用同一可扩展分配器，记录peak_allocated_MiB。原题2.7B/batch4已按用户授权替换，不能标记为原规模结果。

## (b) FP32

单位：MiB（字节/1024²）。forward表示inference/no_grad。train包含前向、loss、反向和自定义AdamW更新。

| 模式 | 128 | 256 | 512 |
|---|---:|---:|---:|
| forward | 3824.171 | 3840.540 | 3907.067 |
| train | 15572.358 | 15528.899 | 15829.898 |

## (c) BF16 autocast混合精度

单位：MiB（字节/1024²）。forward表示inference/no_grad。train包含前向、loss、反向和自定义AdamW更新。

| 模式 | 128 | 256 | 512 |
|---|---:|---:|---:|
| forward | 5643.022 | 5646.376 | 5654.060 |
| train | 15513.659 | 15489.391 | 16248.008 |

## 混合精度的影响

| context | 推理节省比例/% | 训练节省比例/% |
|---|---:|---:|
| 128 | -47.562 | 0.377 |
| 256 | -47.020 | 0.254 |
| 512 | -44.714 | -2.641 |

负值表示BF16显存更多。BF16推理峰值为5643.022–5654.060MiB，高于FP32的3824.171–3907.067MiB；logits类型已验证为BF16，并非开关失效。完整训练在128、256下仅略省显存，在512下反而增加约2.64%，本实现中混合精度没有显著降低训练峰值。FP32参数、梯度和AdamW状态仍占大量显存，autocast权重转换缓存及临时张量也可能增加占用，因此不能假定显存减半。

GPU：RTX3090；PyTorch2.9.1+cu128。128/256训练峰值不是严格递增：此表测的是整个训练步骤的最大值，不仅是前向激活；取峰位置、临时分配和分配器对齐/复用也会影响结果，数据按实测保留。


# 2.7B配置：batch=1，仅推理实测

配置：d_model=2560、d_ff=10240、32层、32头、vocab=10000；context=128/256/512；model.eval()+no_grad；参数保持FP32，BF16使用默认缓存策略的autocast；无优化器、无Profiler。RTX3090，PyTorch2.9.1+cu128，TF32关闭。预热1步后单独测1步显存峰值，再测10次前向耗时取均值和样本标准差。每组独立进程。

| context | FP32峰值/MiB | BF16峰值/MiB | FP32前向均值/ms | BF16前向均值/ms | BF16加速比 |
|---:|---:|---:|---:|---:|---:|
| 128 | 13153.425 | 19586.194 | 106.985 | 115.692 | 0.925x |
| 256 | 13177.438 | 19590.668 | 112.983 | 116.688 | 0.968x |
| 512 | 13286.240 | 19600.014 | 225.499 | 123.858 | 1.821x |

BF16 logits类型已验证，六组均未OOM。BF16推理峰值均高于FP32；参数仍为FP32，autocast转换缓存及临时张量可能增加显存，不能把混合精度等同于纯BF16权重。短上下文128/256下BF16略慢，512下加速约1.82倍；这是当前模型/硬件/实现的实测，不能外推所有配置。

**参数量说明：**按题目称为2.7B的超参数配置运行，但当前BasicsTransformerLM实现实际参数计数为3,406,809,600（约34.07亿），因此报告应同时注明配置和实际实现，不能改写成正好27亿。

| context | FP32前向标准差/ms | BF16前向标准差/ms |
|---:|---:|---:|
| 128 | 0.467 | 0.597 |
| 256 | 1.458 | 0.745 |
| 512 | 1.554 | 0.643 |
